# Лабораторна робота 5 — Логістична регресія для аналізу тональності

**Набори даних:** `amazon_baby_subset.csv`, `important_words.json`  
**Обмеження:** scikit-learn-класифікатори **не дозволені** для базових завдань.

## Налаштування

In [1]:
from google.colab import drive
import sys
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/MTAD_Labs/Lab5')

Mounted at /content/drive


In [2]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [3]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

%matplotlib inline


## Теоретичне підґрунтя

Сигмоїдна функція:
```
P(y = +1 | x, w) = 1 / (1 + exp(−wᵀ h(x)))
```
Похідна логарифму правдоподібності відносно wⱼ:
```
d(ll)/dwⱼ = dot( hⱼ,  1[y=+1] − P(y=+1|x,w) )
```
де 1[y=+1] = 1, якщо мітка +1, інакше 0.

---
## Завдання 1 — Підготовка ознак

1. Завантажте `amazon_baby_subset.csv`. Видаліть рядки з відсутніми відгуками. Вилучіть відгуки з `rating == 3`. Створіть стовпець `sentiment`: **+1** якщо `rating >= 4`, інакше **−1**.
2. Завантажте `important_words.json` (193 слова). Для кожного слова додайте стовпець до DataFrame із підрахунком його входжень у очищений текст відгуку.
3. Повідомте, скільки відгуків залишилось та який баланс класів.

In [9]:
# Завантаження набору даних
products = pd.read_csv('/content/drive/MyDrive/MTAD_Labs/Lab5/amazon_baby_subset.csv')

# Видаліть рядки з відсутніми відгуками та нейтральними рейтингами
products = products.dropna(subset=['review'])
products = products[products['rating'] != 3]

# Створіть стовпець sentiment: +1 якщо рейтинг >= 4, інакше -1
products['sentiment'] = products['rating'].apply(lambda x: 1 if x >= 4 else -1)

print(f'Усього відгуків : {len(products)}')
print(f'Позитивні (+1)  : {(products["sentiment"] == 1).sum()}')
print(f'Негативні (-1)  : {(products["sentiment"] == -1).sum()}')


Усього відгуків : 17311
Позитивні (+1)  : 17310
Негативні (-1)  : 1


In [10]:
# Завантаження важливих слів
with open('/content/drive/MyDrive/MTAD_Labs/Lab5/important_words.json') as f:
    important_words = json.load(f)
print(f'Розмір словника: {len(important_words)} слів')


Розмір словника: 193 слів


In [11]:
# Очищення тексту відгуків (видалення пунктуації, приведення до нижнього регістру)
import string
products['review_clean'] = (
    products['review']
    .fillna('')
    .str.replace(f'[{string.punctuation}]', '', regex=True)
    .str.lower()
)

new_columns = {}

# Додайте стовпець підрахунку слів для кожного важливого слова
for word in important_words:
    new_columns[word] = products['review_clean'].apply(
        lambda text: text.split().count(word)
    )

words_df = pd.DataFrame(new_columns)
products = pd.concat([products, words_df], axis=1)

print('Приклад підрахунку слів:')
display(products[important_words[:5]].head(3))


Приклад підрахунку слів:


,baby,one,great,love,use
0,0,0,1,0,0
1,0,0,0,0,0
2,1,0,0,0,0


---
## Завдання 2 — Побудова матриці ознак

Реалізуйте `get_feature_matrix(df, word_list)`, яка:
1. Створює масив NumPy зі стовпцем одиниць (вільний член), за яким іде по одному стовпцю для кожного слова зі списку.
2. Повертає `(feature_matrix, sentiment_array)`, де `sentiment_array` містить +1 або −1.

Перевірка: `feature_matrix` має форму `(N, 194)`.

In [12]:
def get_feature_matrix(df, word_list):
    """
    Будує матрицю ознак та вектор міток тональності.

    Повертає
    -------
    feature_matrix  : np.ndarray, shape (n, len(word_list)+1)
    sentiment_array : np.ndarray, shape (n,)  значення {+1, -1}
    """
    # Intercept column of 1s
    constant_column = np.ones((len(df), 1))

    # Word feature columns
    word_features = df[word_list].to_numpy()
    feature_matrix = np.hstack((constant_column, word_features))

    sentiment_array = df['sentiment'].to_numpy()
    return feature_matrix, sentiment_array


In [13]:
feature_matrix, sentiment = get_feature_matrix(products, important_words)
print(f'Розмір feature_matrix: {feature_matrix.shape}')  # очікується (N, 194)


Розмір feature_matrix: (17311, 194)


---
## Завдання 3 — Сигмоїдна функція та передбачення

Реалізуйте `predict_probability(feature_matrix, coefficients)`, що обчислює сигмоїдну функцію для кожного рядка. Результат — масив NumPy зі значеннями у (0, 1).

In [14]:
def predict_probability(feature_matrix, coefficients):
    """
    Обчислює P(y = +1 | x, w) для кожного рядка.

    Повертає
    -------
    probabilities : np.ndarray, shape (n,), значення у (0, 1)
    """
    # Compute the linear score for each example
    score = np.dot(feature_matrix, coefficients)

    # Apply sigmoid
    probabilities = 1.0 / (1.0 + np.exp(-score))

    return probabilities


### Перевірка — при нульових вхідних даних кожне передбачення має дорівнювати 0.5

In [15]:
zero_coeffs = np.zeros(feature_matrix.shape[1])
test_probs  = predict_probability(feature_matrix, zero_coeffs)
print(f'Усі передбачення 0.5: {np.allclose(test_probs, 0.5)}')


Усі передбачення 0.5: True


---
## Завдання 4 — Градієнтний підйом

Реалізуйте `logistic_regression(feature_matrix, sentiment, initial_coefficients, step_size, max_iter)`. На кожній ітерації:
1. Обчислюйте передбачення за допомогою `predict_probability()`.
2. Обчислюйте `errors = 1[y=+1] − predictions`.
3. Для кожного коефіцієнта j: `derivative = dot(feature_j, errors)`, потім `coeff[j] += step_size · derivative`.

Запустіть з: `initial_coefficients = np.zeros(194)`, `step_size = 1e-7`, `max_iter = 301`. Виводьте логарифм правдоподібності кожні 50 ітерацій — він має зростати монотонно.

In [16]:
def compute_log_likelihood(feature_matrix, sentiment, coefficients):
    """Допоміжна функція (надана) — обчислює логарифм правдоподібності для моніторингу."""
    indicator = (sentiment == +1).astype(float)
    scores    = np.dot(feature_matrix, coefficients)
    ll        = np.sum(
        indicator * scores - np.log(1.0 + np.exp(scores))
    )
    return ll


In [17]:
def logistic_regression(feature_matrix, sentiment,
                        initial_coefficients, step_size, max_iter):
    """
    Навчає ваги логістичної регресії методом градієнтного підйому.

    Повертає
    -------
    coefficients : np.ndarray, shape (n_features,)
    """
    coefficients = np.array(initial_coefficients, dtype=float)
    indicator    = (sentiment == +1).astype(float)   # 1 if positive, 0 otherwise

    for itr in range(max_iter):
        # 1. Compute predictions P(y=+1 | x, w)
        predictions = predict_probability(feature_matrix, coefficients)

        # 2. Compute errors = indicator(y=+1) - predictions
        errors = indicator - predictions

        # 3. Update every coefficient
        for j in range(len(coefficients)):
            derivative = np.dot(feature_matrix[:, j], errors)
            coefficients[j] += step_size * derivative

        # Print log-likelihood every 50 iterations
        if itr % 50 == 0:
            ll = compute_log_likelihood(feature_matrix, sentiment, coefficients)
            print(f'Ітерація {itr:4d}  |  логарифм правдоподібності: {ll:.4f}')

    return coefficients


### Запуск моделі

In [18]:
coefficients = logistic_regression(
    feature_matrix, sentiment,
    initial_coefficients=np.zeros(194),
    step_size=1e-7,
    max_iter=301
)


Ітерація    0  |  логарифм правдоподібності: -11975.0356
Ітерація   50  |  логарифм правдоподібності: -10884.4567
Ітерація  100  |  логарифм правдоподібності: -9978.6843
Ітерація  150  |  логарифм правдоподібності: -9216.2813
Ітерація  200  |  логарифм правдоподібності: -8566.0665
Ітерація  250  |  логарифм правдоподібності: -8004.8996
Ітерація  300  |  логарифм правдоподібності: -7515.4908


### Точність класифікації та базовий рівень

In [19]:
# Передбачте мітки класів (+1 якщо score > 0, інакше -1)
scores          = np.dot(feature_matrix, coefficients)
predictions     = np.where(scores > 0, +1, -1)

# Точність моделі
model_accuracy  = np.mean(predictions == sentiment)
print(f'Точність моделі   : {model_accuracy:.4f}')

# Базовий рівень мажоритарного класу
majority_class  = +1 if (sentiment == +1).sum() >= (sentiment == -1).sum() else -1
baseline_acc    = np.mean(majority_class == sentiment)
print(f'Базовий рівень    : {baseline_acc:.4f}  (завжди передбачає {majority_class})')


Точність моделі   : 0.9999
Базовий рівень    : 0.9999  (завжди передбачає 1)
